In [8]:
vectorizer_model = CountVectorizer(
    stop_words = STOPWORDS,
    min_df=0.005,
    max_df=0.7,
    ngram_range=(1, 1)
)

In [ ]:
def compute_coherence(topic_model, docs):
    topics = [
        [word for word, _ in topic_model.get_topic(t)]
        for t in topic_model.get_topics().keys() if t != -1
    ]
    tokenized_docs = [doc.split() for doc in docs]
    dictionary = Dictionary(tokenized_docs)
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    return coherence_model.get_coherence()


# ==== Fonction Topic Diversity ====
def compute_topic_diversity(topic_model, topk=10):
    topics = [
        [word for word, _ in topic_model.get_topic(t)[:topk]]
        for t in topic_model.get_topics().keys()
        if t != -1
    ]
    all_words = [word for topic in topics for word in topic]
    unique_words = set(all_words)
    diversity = len(unique_words) / len(all_words) if len(all_words) > 0 else 0
    return diversity


def build_topic_model(n_neighbors, n_components, min_cluster_size=10, min_dist=0.0):
    # UMAP
    umap_model = umap.UMAP(
        n_neighbors=n_neighbors,
        n_components=n_components,
        min_dist=min_dist,
        metric='cosine',
        random_state=42
    )

    # HDBSCAN
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=None,
        metric='euclidean',
        cluster_selection_method='eom'
    )

    # BERTopic
    return BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        top_n_words=10,
        nr_topics=None,
        verbose=False
    )



In [ ]:
n_components_list = [2,3,4]
n_neighbors_list = range(10,20)
min_cluster_size_list = range(10,20)
min_dist_list = [0.1,0.3,0.5] 

results = []
models = {}

# ==== Grid Search ====
for n_comp, n_nbr, min_size, min_d in product(
        n_components_list, n_neighbors_list, min_cluster_size_list, min_dist_list):
    
    tm = build_topic_model(
        n_neighbors=n_nbr,
        n_components=n_comp,
        min_cluster_size=min_size,
        min_dist=min_d
    )
    
    topics, probs = tm.fit_transform(docs, emb)
    
    topic_info = tm.get_topic_info()
    topic0_count = topic_info.loc[topic_info['Topic'] == 0, 'Count'].values[0] if 0 in topic_info['Topic'].values else 0
    
    coh = compute_coherence(tm, docs)
    div = compute_topic_diversity(tm)
    
    score = coh * div
    
    key = (n_comp, n_nbr, min_size, min_d)
    models[key] = (tm, topics, probs)
    
    results.append({
        "n_components": n_comp,
        "n_neighbors": n_nbr,
        "min_cluster_size": min_size,
        "min_dist": min_d,
        "coherence": coh,
        "diversity": div,
        "score": score,
        "n_topics": len(set(topics)) - (1 if -1 in topics else 0),  # nombre de topics réels
        "topic0_count": topic0_count
    })
    
    print(f"comp={n_comp}, neigh={n_nbr}, min_size={min_size}, min_dist={min_d} | "
          f"coh={coh:.4f}, div={div:.4f}, score={score:.4f}, n_topics={len(set(topics))}, "
          f"topic0_count={topic0_count}")

comp=2, neigh=10, min_size=10, min_dist=0.1 | coh=0.3483, div=0.6648, score=0.2316, n_topics=92, topic0_count=220
comp=2, neigh=10, min_size=10, min_dist=0.3 | coh=0.3955, div=0.4917, score=0.1945, n_topics=13, topic0_count=4699
comp=2, neigh=10, min_size=10, min_dist=0.5 | coh=0.4183, div=0.6154, score=0.2574, n_topics=14, topic0_count=4659
comp=2, neigh=10, min_size=11, min_dist=0.1 | coh=0.3561, div=0.6654, score=0.2369, n_topics=79, topic0_count=216
comp=2, neigh=10, min_size=11, min_dist=0.3 | coh=0.3955, div=0.4917, score=0.1945, n_topics=13, topic0_count=4699
comp=2, neigh=10, min_size=11, min_dist=0.5 | coh=0.4119, div=0.4583, score=0.1888, n_topics=13, topic0_count=4659
comp=2, neigh=10, min_size=12, min_dist=0.1 | coh=0.3486, div=0.6439, score=0.2245, n_topics=67, topic0_count=638
comp=2, neigh=10, min_size=12, min_dist=0.3 | coh=0.3991, div=0.5923, score=0.2364, n_topics=14, topic0_count=4651
comp=2, neigh=10, min_size=12, min_dist=0.5 | coh=0.3860, div=0.5000, score=0.1930,

In [11]:
# ==== Transformer en DataFrame ====
df_grid = pd.DataFrame(results)
df_grid = df_grid.sort_values(["coherence"], ascending=False).reset_index(drop=True)

# Afficher le top 10
print(df_grid.head(10))




# Sauvegarder
df_grid.to_csv("grid_search_bertopic4.csv", sep=";", index=False)

   n_components  n_neighbors  min_cluster_size  min_dist  coherence  \
0             2           11                12       0.1   0.509057   
1             3           11                13       0.1   0.480625   
2             2           11                15       0.1   0.477472   
3             2           11                16       0.1   0.477472   
4             2           18                10       0.3   0.476587   
5             2           11                17       0.1   0.475062   
6             2           11                18       0.1   0.475062   
7             2           10                15       0.1   0.474864   
8             2           12                11       0.3   0.474825   
9             4           14                19       0.1   0.472385   

   diversity     score  n_topics  topic0_count  
0   0.525000  0.267255        16          4061  
1   0.500000  0.240312        17          3994  
2   0.557143  0.266020        14          4050  
3   0.557143  0.266020